In [1]:
import osmium

print("osmium imported successfully")

osmium imported successfully


In [5]:
import os
import sys
import pandas as pd
import numpy as np

print("Working directory:")
print(os.getcwd())

print("\nFiles in current directory:")
for f in os.listdir("."):
    print(f)

Working directory:
C:\Users\vinja\Desktop\SIH

Files in current directory:
.ipynb_checkpoints
daily_spatial_dbscan_parameter_test.csv
india-260824.osm.pbf
industrial-context.osm.pbf
industrial-types.osm
industrial-types.osm.pbf
industrial-values.osm.pbf
industrial-works.osm.pbf
industrial-zones.osm.pbf
industrial_tag_analysis.py
modis_2024_India.csv
osm_industrial_profiler.py
SIH_Technical_Blueprint_VIIRS_Industrial_Thermal_POC.pptx
Untitled.ipynb
V0.ipynb
V1.ipynb
V2.ipynb
v3.ipynb
v4.ipynb
v5.ipynb
V6.ipynb
viirs-jpss1_2023_India.csv
viirs-jpss1_2024_India.csv
viirs-snpp_2023_India.csv
viirs-snpp_2024_India.csv
viirs_candidate_episodes_3day.csv
viirs_event_formation_diagnostics.csv
viirs_night_2023_2024_eda.csv
viirs_poc_detection_event_mapping.csv
viirs_poc_event_features.csv
viirs_same_day_spatial_comparison.csv
viirs_spatial_cell_date_gaps.csv
viirs_spatial_cell_temporal_stats.csv
viirs_spatial_grid_density.csv
viirs_spatial_radius_comparison.csv
viirs_spatiotemporal_parameter_com

In [6]:
EVENT_FILE = "viirs_poc_event_features.csv"

if not os.path.exists(EVENT_FILE):
    raise FileNotFoundError(
        f"{EVENT_FILE} not found in {os.getcwd()}"
    )

event_features = pd.read_csv(EVENT_FILE)

event_features["start_date"] = pd.to_datetime(
    event_features["start_date"]
)

event_features["end_date"] = pd.to_datetime(
    event_features["end_date"]
)

print("Events:", len(event_features))
print("Columns:", len(event_features.columns))

display(event_features.head())

Events: 3337
Columns: 23


,event_id,centroid_lat,centroid_lon,spatial_extent_km,start_date,end_date,duration_days,active_days,activity_frequency,detection_count,detections_per_active_day,mean_frp,max_frp,std_frp,frp_range,mean_bright_ti4,max_bright_ti4,std_bright_ti4,ti4_range,mean_bright_ti5,max_bright_ti5,std_bright_ti5,ti5_range
0,1,30.5579,79.1196,0.0614,2024-01-01,2024-01-01,1,1,1.0000,2,2.0000,2.0950,2.2100,0.1626,0.1150,301.0900,304.9900,5.5154,3.9000,277.2550,277.2700,0.0212,0.0150
1,2,30.0443,80.5822,0.0000,2024-01-01,2024-01-01,1,1,1.0000,1,1.0000,1.6600,1.6600,0.0000,0.0000,309.7600,309.7600,0.0000,0.0000,277.5900,277.5900,0.0000,0.0000
2,3,27.4695,95.4220,0.3396,2024-01-01,2024-01-29,29,23,0.7931,33,1.4348,0.9727,2.0200,0.3573,1.0473,305.5497,316.6600,5.6908,11.1103,282.8742,285.1700,1.9206,2.2958
3,4,21.7580,83.8424,0.6851,2024-01-01,2024-01-17,17,17,1.0000,39,2.2941,1.9026,4.1900,0.9092,2.2874,311.6979,328.6800,9.1680,16.9821,288.7890,292.2800,2.4260,3.4910
4,5,21.7341,83.9805,0.0000,2024-01-01,2024-01-01,1,1,1.0000,1,1.0000,1.2100,1.2100,0.0000,0.0000,300.0300,300.0300,0.0000,0.0000,289.4000,289.4000,0.0000,0.0000


In [3]:
import os
import osmium
from collections import Counter

OSM_FILE = r"C:\Users\vinja\Desktop\SIH\industrial-context.osm.pbf"

print("Exists:", os.path.exists(OSM_FILE))
print("Size:", round(os.path.getsize(OSM_FILE) / (1024**2), 2), "MB")

Exists: True
Size: 179.17 MB


In [1]:
import os

OSM_FILE = "industrial-context.osm.pbf"

print("Exists:", os.path.exists(OSM_FILE))
print("Path:", os.path.abspath(OSM_FILE))

if os.path.exists(OSM_FILE):
    print("Size:",
          round(os.path.getsize(OSM_FILE) / (1024**2), 2),
          "MB")

Exists: True
Path: C:\Users\vinja\Desktop\SIH\industrial-context.osm.pbf
Size: 179.17 MB


In [4]:
FILES = [
    r"C:\Users\vinja\Desktop\SIH\industrial-types.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-values.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-works.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-zones.osm.pbf"
]

for f in FILES:
    print(
        os.path.basename(f),
        "->",
        round(os.path.getsize(f) / 1024, 2),
        "KB"
    )

industrial-types.osm.pbf -> 588.06 KB
industrial-values.osm.pbf -> 169.94 KB
industrial-works.osm.pbf -> 325.36 KB
industrial-zones.osm.pbf -> 3016.83 KB


In [5]:
import osmium
from collections import Counter
import os
import time


FILES = [
    r"C:\Users\vinja\Desktop\SIH\industrial-types.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-values.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-works.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-zones.osm.pbf"
]


class SmallOSMInspector(osmium.SimpleHandler):

    def __init__(self):
        super().__init__()

        self.nodes = 0
        self.ways = 0
        self.relations = 0

        self.tags = Counter()

    def process_tags(self, tags):

        for key, value in tags:
            self.tags[(key, value)] += 1

    def node(self, n):
        self.nodes += 1
        self.process_tags(n.tags)

    def way(self, w):
        self.ways += 1
        self.process_tags(w.tags)

    def relation(self, r):
        self.relations += 1
        self.process_tags(r.tags)


for file in FILES:

    print("\n" + "=" * 70)
    print(os.path.basename(file))
    print("=" * 70)

    if not os.path.exists(file):
        print("FILE NOT FOUND")
        continue

    size_kb = os.path.getsize(file) / 1024

    print("Size:", round(size_kb, 2), "KB")
    print("Reading...")

    start = time.time()

    inspector = SmallOSMInspector()
    inspector.apply_file(file, locations=False)

    elapsed = time.time() - start

    print("Nodes     :", f"{inspector.nodes:,}")
    print("Ways      :", f"{inspector.ways:,}")
    print("Relations :", f"{inspector.relations:,}")
    print("Time      :", round(elapsed, 2), "seconds")

    print("\nTop tags:")

    for (key, value), count in inspector.tags.most_common(30):
        print(f"{key:<20} {str(value):<35} {count:,}")


industrial-types.osm.pbf
Size: 588.06 KB
Reading...
Nodes     : 77,070
Ways      : 5,224
Relations : 52
Time      : 8.39 seconds

Top tags:
landuse              industrial                          4,800
industrial           brickyard                           2,800
industrial           depot                               581
source               Esri World Imagery                  525
industrial           factory                             384
source               Esri World imagery (retreived 04/2024) 275
depot                bus                                 265
building             industrial                          255
source               bing                                246
industrial           cooling                             177
industrial           brickworks                          173
barrier              wall                                169
industrial           mine                                150
source               PGS                                 14

In [6]:
import osmium
import pandas as pd
import os
import time


OSM_FILES = [
    r"C:\Users\vinja\Desktop\SIH\industrial-types.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-works.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-zones.osm.pbf"
]


class OSMFeatureExtractor(osmium.SimpleHandler):

    def __init__(self):
        super().__init__()

        self.records = []

    def extract_tags(self, tags):

        return {
            "industrial": tags.get("industrial"),
            "landuse": tags.get("landuse"),
            "power": tags.get("power"),
            "man_made": tags.get("man_made"),
            "building": tags.get("building"),
            "product": tags.get("product"),
            "plant_source": tags.get("plant:source"),
            "plant_method": tags.get("plant:method"),
            "resource": tags.get("resource"),
            "description": tags.get("description")
        }

    def add_record(self, obj_type, obj_id, lat, lon, tags):

        relevant = any([
            tags.get("industrial"),
            tags.get("landuse"),
            tags.get("power"),
            tags.get("man_made")
        ])

        if not relevant:
            return

        row = {
            "osm_type": obj_type,
            "osm_id": obj_id,
            "latitude": lat,
            "longitude": lon
        }

        row.update(self.extract_tags(tags))

        self.records.append(row)

    def node(self, n):

        if n.location.valid():

            self.add_record(
                "node",
                n.id,
                n.location.lat,
                n.location.lon,
                n.tags
            )

    def way(self, w):

        # We do not have geometry here.
        # Ways will be handled separately if needed.
        pass

    def relation(self, r):

        pass


all_features = []

for file in OSM_FILES:

    print("\nReading:", os.path.basename(file))

    start = time.time()

    extractor = OSMFeatureExtractor()
    extractor.apply_file(file, locations=True)

    df = pd.DataFrame(extractor.records)

    print(
        "Extracted:",
        len(df),
        "point features"
    )

    print(
        "Time:",
        round(time.time() - start, 2),
        "seconds"
    )

    if len(df) > 0:
        df["source_file"] = os.path.basename(file)
        all_features.append(df)


osm_points = pd.concat(
    all_features,
    ignore_index=True
)

print("\n" + "=" * 60)
print("TOTAL OSM POINT FEATURES")
print("=" * 60)

print("Rows:", len(osm_points))
print("Columns:", len(osm_points.columns))

display(osm_points.head())


Reading: industrial-types.osm.pbf
Extracted: 171 point features
Time: 1.21 seconds

Reading: industrial-works.osm.pbf
Extracted: 1553 point features
Time: 0.87 seconds

Reading: industrial-zones.osm.pbf
Extracted: 257 point features
Time: 13.54 seconds

TOTAL OSM POINT FEATURES
Rows: 1981
Columns: 15


,osm_type,osm_id,latitude,longitude,industrial,landuse,power,man_made,building,product,plant_source,plant_method,resource,description,source_file
0,node,343703676,10.337727,76.221943,slaughterhouse,industrial,None,None,None,None,None,None,None,None,industrial-types.osm.pbf
1,node,1545193206,23.239580,69.791493,business,None,None,None,None,None,None,None,None,None,industrial-types.osm.pbf
2,node,2527030591,13.238652,80.098155,None,None,tower,None,None,None,None,None,None,None,industrial-types.osm.pbf
3,node,2556368813,10.712002,79.516186,sawmill,None,None,None,None,None,None,None,None,None,industrial-types.osm.pbf
4,node,3358197637,12.970638,74.840782,depot,industrial,None,None,None,None,None,None,None,None,industrial-types.osm.pbf


In [7]:
display(osm_points[
    [
        "industrial",
        "landuse",
        "power",
        "man_made",
        "building",
        "product"
    ]
].notna().sum())

industrial     285
landuse        363
power            8
man_made      1564
building        13
product        326
dtype: int64

In [8]:
osm_points.to_csv(
    "osm_points_v7.csv",
    index=False
)

print("Saved:", "osm_points_v7.csv")
print("Rows:", len(osm_points))

Saved: osm_points_v7.csv
Rows: 1981
